In [3]:
import pandas as pd
df = pd.read_csv("/Users/hiteshbisht/personal/Real-Time-Financial-Risk/src/datasets/notebooks/raw/PS_20174392719_1491204439457_log.csv")
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [4]:
df.shape

(6362620, 11)

In [5]:
df.columns

Index(['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig',
       'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud',
       'isFlaggedFraud'],
      dtype='str')

In [6]:
df["isFraud"].value_counts(normalize=True)

isFraud
0    0.998709
1    0.001291
Name: proportion, dtype: float64

In [7]:
df["type"].value_counts()

type
CASH_OUT    2237500
PAYMENT     2151495
CASH_IN     1399284
TRANSFER     532909
DEBIT         41432
Name: count, dtype: int64

In [8]:
df["step"].min(), df["step"].max()

(np.int64(1), np.int64(743))

In [9]:
train = df[df["step"] <= 500]
test  = df[df["step"] > 500]

train.shape, test.shape

((6061807, 11), (300813, 11))

In [11]:
target = "isFraud"

# drop columns jo direct ID hain (model ke liye useful nahi)
drop_cols = ["isFraud", "nameOrig", "nameDest"]  # IDs

X_train = train.drop(columns=drop_cols)
y_train = train[target]

X_test  = test.drop(columns=drop_cols)
y_test  = test[target]
X_train.shape, X_test.shape

((6061807, 8), (300813, 8))

In [12]:
X_train = pd.get_dummies(X_train, columns=["type"], drop_first=True)
X_test  = pd.get_dummies(X_test,  columns=["type"], drop_first=True)

# ensure same columns in train & test
X_train, X_test = X_train.align(X_test, join="left", axis=1, fill_value=0)

X_train.shape, X_test.shape

((6061807, 11), (300813, 11))

In [13]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [14]:
from sklearn.metrics import average_precision_score, confusion_matrix, classification_report

y_prob = model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

print("PR-AUC:", average_precision_score(y_test, y_prob))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nReport:\n", classification_report(y_test, y_pred))

PR-AUC: 0.8272904292780572
Confusion Matrix:
 [[298145     16]
 [  1320   1332]]

Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00    298161
           1       0.99      0.50      0.67      2652

    accuracy                           1.00    300813
   macro avg       0.99      0.75      0.83    300813
weighted avg       1.00      1.00      0.99    300813



In [16]:
threshold = 0.1
y_pred_t = (y_prob >= threshold).astype(int)

print("Threshold:", threshold)
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_t))
print("\nReport:\n", classification_report(y_test, y_pred_t))

Threshold: 0.1
Confusion Matrix:
 [[297983    178]
 [   938   1714]]

Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00    298161
           1       0.91      0.65      0.75      2652

    accuracy                           1.00    300813
   macro avg       0.95      0.82      0.88    300813
weighted avg       1.00      1.00      1.00    300813



In [17]:
print("Missing values per column:\n")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nFraud count by transaction type:\n")
print(pd.crosstab(df["type"], df["isFraud"]))

print("\nFraud rate by transaction type:\n")
print(df.groupby("type")["isFraud"].mean().sort_values(ascending=False))

print("\nAmount summary by fraud label:\n")
print(df.groupby("isFraud")["amount"].describe())

print("\nFlagged fraud distribution:\n")
print(pd.crosstab(df["isFlaggedFraud"], df["isFraud"]))

Missing values per column:

step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

Duplicate rows: 0

Fraud count by transaction type:

isFraud         0     1
type                   
CASH_IN   1399284     0
CASH_OUT  2233384  4116
DEBIT       41432     0
PAYMENT   2151495     0
TRANSFER   528812  4097

Fraud rate by transaction type:

type
TRANSFER    0.007688
CASH_OUT    0.001840
CASH_IN     0.000000
DEBIT       0.000000
PAYMENT     0.000000
Name: isFraud, dtype: float64

Amount summary by fraud label:

             count          mean           std   min         25%        50%  \
isFraud                                                                       
0        6354407.0  1.781970e+05  5.962370e+05  0.01   13368.395   74684.72   
1           8213.0  1.467967e+06  2.404253e+06  0.00  127091.330  441423.

In [18]:
df["errorBalanceOrig"] = (df["oldbalanceOrg"] - df["newbalanceOrig"]) - df["amount"]
df["errorBalanceDest"] = (df["newbalanceDest"] - df["oldbalanceDest"]) - df["amount"]

print("Feature preview:\n")
print(df[["amount", "errorBalanceOrig", "errorBalanceDest", "isFraud"]].head())

print("\nFeature summary by fraud label:\n")
print(df.groupby("isFraud")[["errorBalanceOrig", "errorBalanceDest"]].describe())

Feature preview:

     amount  errorBalanceOrig  errorBalanceDest  isFraud
0   9839.64      1.455192e-11          -9839.64        0
1   1864.28     -1.136868e-12          -1864.28        0
2    181.00      0.000000e+00           -181.00        1
3    181.00      0.000000e+00         -21363.00        1
4  11668.14      0.000000e+00         -11668.14        0

Feature summary by fraud label:

        errorBalanceOrig                                             \
                   count           mean            std          min   
isFraud                                                               
0              6354407.0 -201338.558109  606928.890826 -92445516.64   
1                 8213.0  -10692.325265  265146.131130 -10000000.00   

                                                     errorBalanceDest  \
               25%       50%       75%           max            count   
isFraud                                                                 
0       -249953.43 -69049.31 -3